# 11.18 — DDPG & TD3

DDPG and TD3 are actor-critic methods for **continuous actions**: instead of choosing from a short action list, the agent outputs a real number such as torque, throttle, or steering. In this lesson, we build a tiny continuous-control world from scratch, fit critics with Bellman targets, update a deterministic actor by differentiating through the critic, and then add TD3's three stabilizers: twin critics, clipped double-Q targets, target-policy smoothing, and delayed actor updates.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build DDPG and TD3 one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is small enough to inspect, including deterministic policy gradients, twin critics, and target smoothing. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, gradients, and small least-squares fits.
import matplotlib.pyplot as plt  # walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy rollouts and noise.

### 1. Continuous action control: a tiny one-step world

A discrete Q-table can try every action in a short list. DDPG is needed when the action is continuous, so trying every possible action is impossible. We start with a one-step toy world: the state is a desired position `s`, the action `a` is the controller's chosen move, and reward is highest when `a` matches `s`. This makes the optimal deterministic policy obvious: choose `a=s`.

In [ ]:
s_w = np.linspace(-1.0, 1.0, 9)                 # desired positions.
a_grid_w = np.linspace(-1.5, 1.5, 121)           # only for visualization; the action space is continuous.
reward_w = -(a_grid_w[None, :] - s_w[:, None]) ** 2  # r(s,a)=-(a-s)^2.
best_actions_w = a_grid_w[np.argmax(reward_w, axis=1)]  # best grid action for each state.
print("states:", np.round(s_w, 2))
print("best grid actions:", np.round(best_actions_w, 2))
assert np.allclose(best_actions_w, s_w)

▶ What you'll see: the best action tracks the state almost exactly, showing the continuous control target.

In [ ]:
plt.figure(figsize=(5, 3))
plt.imshow(reward_w, aspect="auto", origin="lower", extent=[a_grid_w[0], a_grid_w[-1], s_w[0], s_w[-1]], cmap="viridis")
plt.colorbar(label="reward")
plt.plot(s_w, s_w, color="white", linewidth=2, label="optimal a=s")
plt.xlabel("action a"); plt.ylabel("state s"); plt.title("1: reward ridge in continuous action space")
plt.legend(); plt.show()

▶ What you'll see: the bright ridge lies on `a=s`; DDPG tries to climb that ridge without enumerating all actions.

*Why it's done this way:* a continuous action set has infinitely many possible moves, so an argmax over actions is not a table lookup. A deterministic actor `μ(s)` turns the control problem into learning a smooth function from states to actions, while a critic `Q(s,a)` provides a differentiable surface telling the actor which way to move.

### 2. A critic is a differentiable action-value surface

The critic estimates `Q(s,a)`: the discounted consequence of taking action `a` in state `s`. In this one-step world, `Q` equals the reward, so we can fit a simple quadratic critic with NumPy least squares and inspect its action gradient. The feature vector `[1, s, a, s², sa, a²]` is enough to represent the true bowl exactly.

In [ ]:
S_w, A_w = np.meshgrid(np.linspace(-1, 1, 17), np.linspace(-1.5, 1.5, 31), indexing="ij")
y_w = -((A_w - S_w) ** 2).ravel()              # true one-step Q values.
X_w = np.column_stack([np.ones(S_w.size), S_w.ravel(), A_w.ravel(), S_w.ravel() ** 2, S_w.ravel() * A_w.ravel(), A_w.ravel() ** 2])
w_critic_w = np.linalg.lstsq(X_w, y_w, rcond=None)[0]
print("critic weights:", np.round(w_critic_w, 3))
assert np.allclose(np.round(w_critic_w, 3), [0, 0, 0, -1, 2, -1])

▶ What you'll see: the fitted critic recovers `Q(s,a)=−s²+2sa−a²`, exactly the expanded form of `−(a−s)²`.

In [ ]:
def q_quad_w(s, a, w):
    return w[0] + w[1]*s + w[2]*a + w[3]*s*s + w[4]*s*a + w[5]*a*a

def dq_da_w(s, a, w):
    return w[2] + w[4]*s + 2*w[5]*a

test_s_w, test_a_w = 0.7, 0.1
print("Q(0.7,0.1):", round(float(q_quad_w(test_s_w, test_a_w, w_critic_w)), 3))
print("dQ/da at (0.7,0.1):", round(float(dq_da_w(test_s_w, test_a_w, w_critic_w)), 3))
assert round(float(dq_da_w(test_s_w, test_a_w, w_critic_w)), 3) == 1.2

▶ What you'll see: when action `0.1` is below the best action `0.7`, the critic gradient is positive, telling the actor to increase the action.

In [ ]:
a_line_w = np.linspace(-1.0, 1.4, 100)
q_line_w = q_quad_w(test_s_w, a_line_w, w_critic_w)
plt.figure(figsize=(5, 3))
plt.plot(a_line_w, q_line_w, color="purple")
plt.axvline(test_a_w, color="red", linestyle="--", label="current action")
plt.axvline(test_s_w, color="green", linestyle="--", label="best action")
plt.xlabel("action a"); plt.ylabel("Q(s,a)"); plt.title("2: critic surface slice")
plt.legend(); plt.show()

▶ What you'll see: a parabola peaking at `a=s`; the local slope at the current action points toward the peak.

*Why it's done this way:* the critic replaces an impossible continuous argmax with a local derivative. If `∂Q/∂a` is positive, a slightly larger action has better predicted consequence; if it is negative, a smaller action is better. DDPG works because the actor can follow this critic-provided gradient.

### 3. Deterministic policy gradient: update the actor through the critic

A deterministic actor outputs one action per state: `μθ(s)=θs` in this tiny example. The objective is `J(θ)=E_s[Q(s, μθ(s))]`. By the chain rule,

$$\nabla_\theta J=\mathbb{E}_s\left[\nabla_a Q(s,a)\vert_{a=\mu_\theta(s)}\,\nabla_\theta\mu_\theta(s)\right].$$

So the actor is not given a target action directly; it receives the critic's action slope and asks how changing θ would change the action.

In [ ]:
theta_w = 0.25                                      # current actor: action = 0.25*s.
states_pg_w = np.array([-1.0, -0.5, 0.5, 1.0])       # symmetric states.
a_pg_w = theta_w * states_pg_w                       # actor actions.
slopes_pg_w = dq_da_w(states_pg_w, a_pg_w, w_critic_w) # critic dQ/da at actor actions.
grad_theta_w = float(np.mean(slopes_pg_w * states_pg_w)) # dμ/dθ=s.
print("actions:", np.round(a_pg_w, 3))
print("dQ/da:", np.round(slopes_pg_w, 3))
print("deterministic policy gradient:", round(grad_theta_w, 3))
assert abs(grad_theta_w - 0.9375) < 1e-9

▶ What you'll see: θ is too small, so actions under-react to states and the gradient is positive.

In [ ]:
eta_actor_w = 0.2
new_theta_w = theta_w + eta_actor_w * grad_theta_w
old_obj_w = float(np.mean(q_quad_w(states_pg_w, theta_w * states_pg_w, w_critic_w)))
new_obj_w = float(np.mean(q_quad_w(states_pg_w, new_theta_w * states_pg_w, w_critic_w)))
print("theta before -> after:", round(theta_w, 3), "->", round(new_theta_w, 3))
print("objective before -> after:", round(old_obj_w, 3), "->", round(new_obj_w, 3))
assert new_obj_w > old_obj_w

▶ What you'll see: one policy-gradient step moves θ closer to 1 and raises the mean Q value.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(states_pg_w, states_pg_w, "k--", label="optimal a=s")
plt.plot(states_pg_w, theta_w * states_pg_w, "o-", label="before")
plt.plot(states_pg_w, new_theta_w * states_pg_w, "o-", label="after")
plt.xlabel("state s"); plt.ylabel("actor action μ(s)"); plt.title("3: actor update follows critic gradient")
plt.legend(); plt.show()

▶ What you'll see: the actor line rotates toward the optimal line after a single gradient ascent step.

*Why it's done this way:* deterministic policy gradients avoid sampling many actions from a stochastic policy. The critic supplies `∂Q/∂a`, and the actor supplies `∂μ/∂θ`; multiplying them is exactly the chain rule for how θ changes long-run value through its chosen action.

### 4. DDPG stabilizers: replay, bootstrapped targets, and slow target networks

In real tasks, the critic is trained from transitions `(s,a,r,s')`, not from known true Q values. DDPG uses a replay buffer to reuse off-policy transitions and a slow-moving target actor/critic to compute the Bellman target `y=r+γQ_target(s', μ_target(s'))`. The target network is a low-pass filter: it changes slowly so the critic is not chasing a target that jumps every step.

In [ ]:
rng_w = np.random.default_rng(1)
replay_s_w = rng_w.uniform(-1, 1, size=8)
replay_a_w = np.clip(replay_s_w + rng_w.normal(0, 0.25, size=8), -1.5, 1.5)
replay_sp_w = 0.5 * replay_s_w + 0.5 * replay_a_w
replay_r_w = -(replay_a_w - replay_s_w) ** 2 - 0.1 * replay_sp_w ** 2
gamma_w = 0.9
print("first transition:", np.round([replay_s_w[0], replay_a_w[0], replay_r_w[0], replay_sp_w[0]], 3))
assert replay_s_w.shape == replay_a_w.shape == replay_sp_w.shape

▶ What you'll see: a small replay buffer of continuous states, actions, rewards, and next states.

In [ ]:
theta_online_w, theta_target_w = 0.8, 0.5
w_online_w = w_critic_w.copy()
w_target_w = 0.8 * w_critic_w + np.array([0.05, 0.02, -0.03, 0.0, 0.1, -0.05])
next_actions_w = theta_target_w * replay_sp_w
target_q_w = q_quad_w(replay_sp_w, next_actions_w, w_target_w)
y_ddpg_w = replay_r_w + gamma_w * target_q_w
print("first 3 Bellman targets:", np.round(y_ddpg_w[:3], 3))
assert y_ddpg_w.shape == replay_r_w.shape

▶ What you'll see: each target combines immediate reward with the target critic's estimate of the next state-action consequence.

In [ ]:
tau_w = 0.1
old_target_theta_w = theta_target_w
theta_target_w = tau_w * theta_online_w + (1 - tau_w) * theta_target_w
print("target actor θ:", round(old_target_theta_w, 3), "->", round(theta_target_w, 3))
assert round(theta_target_w, 3) == 0.53

▶ What you'll see: the target actor moves only 10% of the way toward the online actor.

*Why it's done this way:* bootstrapping is powerful but dangerous because the learner trains against its own estimates. Replay breaks short-term correlation in data, and target networks make the right-hand side of the Bellman equation drift slowly instead of vibrating with every critic update.

### 5. TD3 twin critics: clipped double-Q reduces overestimation

DDPG can overestimate because the actor seeks actions with high critic values, including accidental critic errors. TD3 trains two critics and uses the smaller prediction in the target: `min(Q1,Q2)`. This is a pessimistic backup. It may be biased low, but it blocks the actor from exploiting one critic's optimistic mistake.

In [ ]:
a_td3_w = np.linspace(-1.2, 1.2, 121)
s_td3_w = 0.4
true_q_td3_w = -(a_td3_w - s_td3_w) ** 2
q1_td3_w = true_q_td3_w + 0.28 * np.exp(-((a_td3_w - 0.95) / 0.18) ** 2)  # optimistic bump.
q2_td3_w = true_q_td3_w - 0.05 + 0.04 * np.sin(5 * a_td3_w)                # independent critic.
min_q_td3_w = np.minimum(q1_td3_w, q2_td3_w)
print("max Q1 action:", round(float(a_td3_w[np.argmax(q1_td3_w)]), 2))
print("max min(Q1,Q2) action:", round(float(a_td3_w[np.argmax(min_q_td3_w)]), 2))
assert abs(float(a_td3_w[np.argmax(min_q_td3_w)]) - s_td3_w) < 0.12

▶ What you'll see: Q1 is tempted by an optimistic bump, while the clipped minimum stays near the real optimum.

In [ ]:
plt.figure(figsize=(5.2, 3))
plt.plot(a_td3_w, true_q_td3_w, "k--", label="true Q")
plt.plot(a_td3_w, q1_td3_w, label="critic 1")
plt.plot(a_td3_w, q2_td3_w, label="critic 2")
plt.plot(a_td3_w, min_q_td3_w, linewidth=2, label="min(Q1,Q2)")
plt.xlabel("action a"); plt.ylabel("Q estimate"); plt.title("5: clipped double-Q target")
plt.legend(); plt.show()

▶ What you'll see: the minimum curve refuses to inherit critic 1's isolated optimistic spike.

*Why it's done this way:* maximization amplifies positive noise: if one action has an erroneously high value, the actor will move toward it. Taking the minimum of two critics is a simple bias-variance trade: accept some pessimism to remove the high-side errors that deterministic actors are especially good at exploiting.

### 6. TD3 target smoothing and delayed actor updates

TD3 also smooths target actions by adding clipped noise before evaluating the target critic, and it updates the actor less often than the critics. Smoothing asks the target to be good in a small neighborhood of the action, not just at one brittle point. Delaying the actor gives the critics time to become less wrong before the policy follows their gradients.

In [ ]:
rng_smooth_w = np.random.default_rng(2)
base_next_action_w = 0.6
noise_w = np.clip(rng_smooth_w.normal(0, 0.2, size=2000), -0.3, 0.3)
smoothed_actions_w = np.clip(base_next_action_w + noise_w, -1.0, 1.0)
print("noise mean/std:", round(float(noise_w.mean()), 3), round(float(noise_w.std()), 3))
print("smoothed action range:", round(float(smoothed_actions_w.min()), 2), round(float(smoothed_actions_w.max()), 2))
assert smoothed_actions_w.min() >= -1 and smoothed_actions_w.max() <= 1

▶ What you'll see: noise is small, clipped, and action-bounded, so targets are locally averaged without leaving valid action limits.

In [ ]:
def spiky_q_w(a):
    return -((a - 0.4) ** 2) + 0.7 * np.exp(-((a - 0.62) / 0.035) ** 2)

point_target_w = float(spiky_q_w(base_next_action_w))
smooth_target_w = float(np.mean(spiky_q_w(smoothed_actions_w)))
print("point target Q:", round(point_target_w, 3))
print("smoothed target Q:", round(smooth_target_w, 3))
assert smooth_target_w < point_target_w

▶ What you'll see: the point estimate is inflated by a narrow spike, while noise-averaging lowers the target.

In [ ]:
updates_w = np.arange(1, 9)
critic_updates_w = np.ones_like(updates_w)
actor_updates_w = (updates_w % 2 == 0).astype(int)
print("actor update flags:", actor_updates_w)
plt.figure(figsize=(5, 2.7))
plt.step(updates_w, np.cumsum(critic_updates_w), where="mid", label="critic updates")
plt.step(updates_w, np.cumsum(actor_updates_w), where="mid", label="actor updates")
plt.xlabel("training step"); plt.ylabel("cumulative updates"); plt.title("6: delayed actor updates")
plt.legend(); plt.show()

▶ What you'll see: the critic moves every step, while the actor moves every other step.

*Why it's done this way:* target smoothing penalizes value spikes that disappear under tiny action perturbations, and delayed policy updates reduce feedback loops where a weak critic misleads the actor, the actor shifts the data, and the critic gets even less reliable.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, random numbers, least-squares critics, and hand-coded gradients.
import matplotlib.pyplot as plt # load Matplotlib so policies, critics, targets, and learning curves can be inspected visually.
np.random.seed(0) # make all random examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Continuous actions are real-valued

**Goal.** Build the smallest continuous-action reward curve, because DDPG and TD3 choose real-valued actions rather than indices from a table. We build it in 2 steps.

In [ ]:
s_b1 = 0.6 # define one state as the target position the action should match.
a_b1 = np.linspace(-1.0, 1.2, 80) # create a dense action grid only for visualization.
r_b1 = -((a_b1 - s_b1) ** 2) # compute reward as negative squared action error.
print("best grid action:", round(float(a_b1[np.argmax(r_b1)]), 3)) # inspect the action with highest reward on the display grid.

▶ What you'll see: the best displayed action is close to the state value 0.6.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact reward-curve figure.
plt.plot(a_b1, r_b1, color="teal") # draw reward for many possible continuous actions.
plt.axvline(s_b1, color="black", linestyle="--", label="true best a=s") # mark the analytic optimum.
plt.title("Basic 1: continuous action reward") # title the plot.
plt.xlabel("action a") # label the action axis.
plt.ylabel("reward") # label the reward axis.
plt.legend() # show the optimum label.
plt.show() # display the curve.

▶ What you'll see: a smooth parabola whose peak is at the action matching the state.

👀 Takeaway: continuous-control policies need to output real numbers, not action IDs.

### Basic 2 — A deterministic actor maps state to action

**Goal.** Define `μ(s)=θs`, because DDPG's actor is a deterministic function from state to continuous action. We build it in 2 steps.

In [ ]:
theta_b2 = 0.5 # choose a simple actor parameter.
states_b2 = np.array([-1.0, -0.5, 0.0, 0.5, 1.0]) # define several states to feed the actor.
actions_b2 = theta_b2 * states_b2 # compute deterministic actor outputs.
print("actor actions:", np.round(actions_b2, 3)) # inspect the real-valued actions.

▶ What you'll see: every state maps to exactly one action, scaled by θ.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact policy plot.
plt.plot(states_b2, actions_b2, "o-", label="μ(s)=0.5s") # draw the actor line.
plt.plot(states_b2, states_b2, "k--", label="optimal μ(s)=s") # draw the optimal line.
plt.title("Basic 2: deterministic actor") # title the plot.
plt.xlabel("state s") # label the state axis.
plt.ylabel("action μ(s)") # label the action axis.
plt.legend() # show policy labels.
plt.show() # display the policy plot.

▶ What you'll see: the actor has the right sign but under-reacts to each state.

👀 Takeaway: the actor is a smooth controller whose parameters decide which action each state receives.

### Basic 3 — Compute one critic value

**Goal.** Evaluate `Q(s,a)=−(a−s)^2`, because the critic scores how good a chosen continuous action is in a state. We build it in 2 steps.

In [ ]:
s_b3 = 0.8 # choose one state.
a_b3 = 0.2 # choose one candidate action.
q_b3 = -((a_b3 - s_b3) ** 2) # compute the one-step action value.
print("Q(s,a):", round(q_b3, 3)) # inspect the critic value.
assert round(q_b3, 3) == -0.36 # verify the worked number.

▶ What you'll see: the value is negative because the action misses the state by 0.6.

In [ ]:
a_compare_b3 = np.array([0.2, 0.5, 0.8]) # choose three actions to compare.
q_compare_b3 = -((a_compare_b3 - s_b3) ** 2) # evaluate the critic at all three actions.
plt.figure(figsize=(4, 3)) # create a compact comparison plot.
plt.bar(["0.2", "0.5", "0.8"], q_compare_b3, color="orange") # show critic scores by action.
plt.title("Basic 3: critic values") # title the bar chart.
plt.ylabel("Q(0.8,a)") # label the value axis.
plt.show() # display the chart.

▶ What you'll see: the action equal to the state has the highest value.

👀 Takeaway: a critic turns each state-action pair into a scalar consequence estimate.

### Basic 4 — Differentiate the critic with respect to action

**Goal.** Compute `∂Q/∂a`, because deterministic actors improve by following the critic's action slope. We build it in 2 steps.

In [ ]:
s_b4 = 0.8 # choose a state where the best action is 0.8.
a_b4 = 0.2 # choose an action below the optimum.
dq_da_b4 = -2 * (a_b4 - s_b4) # differentiate Q=-(a-s)^2 with respect to action.
print("dQ/da:", round(dq_da_b4, 3)) # inspect the slope.
assert round(dq_da_b4, 3) == 1.2 # verify the worked gradient.

▶ What you'll see: the slope is positive, so increasing the action should improve Q.

In [ ]:
a_line_b4 = np.linspace(0.0, 1.2, 100) # create action values around the optimum.
q_line_b4 = -((a_line_b4 - s_b4) ** 2) # evaluate the critic curve.
plt.figure(figsize=(4, 3)) # create a compact slope plot.
plt.plot(a_line_b4, q_line_b4, color="purple") # draw the critic curve.
plt.scatter([a_b4], [-((a_b4 - s_b4) ** 2)], color="red", label="current action") # mark the current action.
plt.title("Basic 4: critic slope points uphill") # title the plot.
plt.xlabel("action a") # label the action axis.
plt.ylabel("Q") # label the value axis.
plt.legend() # show marker label.
plt.show() # display the curve.

▶ What you'll see: the current action sits on the left side of the peak, where the curve slopes upward.

👀 Takeaway: `∂Q/∂a` tells the actor which direction to move its continuous output.

### Basic 5 — Chain rule for one policy-gradient number

**Goal.** Multiply `∂Q/∂a` by `∂μ/∂θ`, because deterministic policy gradients are ordinary chain rule gradients. We build it in 2 steps.

In [ ]:
s_b5 = 0.8 # choose the state.
theta_b5 = 0.25 # choose a too-small actor parameter.
a_b5 = theta_b5 * s_b5 # compute the actor action.
dq_da_b5 = -2 * (a_b5 - s_b5) # compute critic slope at the actor action.
dmu_dtheta_b5 = s_b5 # derivative of μ=θs with respect to θ.
grad_b5 = dq_da_b5 * dmu_dtheta_b5 # apply the chain rule.
print("a:", round(a_b5, 3), "dQ/da:", round(dq_da_b5, 3), "gradient:", round(grad_b5, 3)) # inspect the pieces.
assert round(grad_b5, 3) == 0.96 # verify the worked number.

▶ What you'll see: the policy gradient is positive because θ should increase.

In [ ]:
new_theta_b5 = theta_b5 + 0.1 * grad_b5 # take one small gradient-ascent step on θ.
print("theta before -> after:", round(theta_b5, 3), "->", round(new_theta_b5, 3)) # inspect the actor update.
plt.figure(figsize=(4, 3)) # create a before-after action plot.
plt.bar(["before", "after", "optimal"], [theta_b5*s_b5, new_theta_b5*s_b5, s_b5], color=["gray", "teal", "green"]) # compare actions.
plt.title("Basic 5: chain-rule actor update") # title the chart.
plt.ylabel("action for s=0.8") # label the action scale.
plt.show() # display the chart.

▶ What you'll see: the actor action moves closer to the optimal action.

👀 Takeaway: DDPG's actor update is the critic's action gradient passed backward through the actor.

### Basic 6 — Add exploration noise to a deterministic actor

**Goal.** Add noise during data collection, because a deterministic actor repeats the same action unless exploration perturbs it. We build it in 2 steps.

In [ ]:
rng_b6 = np.random.default_rng(6) # create local reproducible randomness.
s_b6 = np.ones(200) * 0.4 # repeat one state many times.
theta_b6 = 0.7 # choose a deterministic actor parameter.
noise_b6 = rng_b6.normal(0, 0.15, size=s_b6.size) # draw Gaussian exploration noise.
a_noisy_b6 = np.clip(theta_b6 * s_b6 + noise_b6, -1, 1) # add noise and keep actions within valid bounds.
print("mean action:", round(float(a_noisy_b6.mean()), 3), "std:", round(float(a_noisy_b6.std()), 3)) # inspect noisy behavior.

▶ What you'll see: actions cluster around the deterministic value but vary enough to explore.

In [ ]:
plt.figure(figsize=(4, 3)) # create a histogram of exploratory actions.
plt.hist(a_noisy_b6, bins=20, color="steelblue", edgecolor="white") # visualize the noisy action distribution.
plt.axvline(theta_b6 * 0.4, color="black", linestyle="--", label="μ(s)") # mark the deterministic action.
plt.title("Basic 6: exploration noise") # title the plot.
plt.xlabel("sampled action") # label sampled actions.
plt.ylabel("count") # label frequency.
plt.legend() # show deterministic marker.
plt.show() # display the histogram.

▶ What you'll see: a cloud of actions around `μ(s)`, not a single repeated action.

👀 Takeaway: DDPG explores by perturbing deterministic actions while storing the resulting transitions off-policy.

### Basic 7 — Build one Bellman target

**Goal.** Compute `y=r+γQ(s', μ(s'))`, because the critic learns from bootstrapped targets. We build it in 2 steps.

In [ ]:
r_b7 = 1.0 # observed immediate reward.
gamma_b7 = 0.9 # discount future value.
q_next_b7 = 0.8 # target critic's estimate at the next state and target action.
y_b7 = r_b7 + gamma_b7 * q_next_b7 # one-step Bellman target.
print("Bellman target:", round(y_b7, 3)) # inspect the target.
assert round(y_b7, 3) == 1.72 # verify the canonical lesson number.

▶ What you'll see: the target is reward plus discounted next estimate.

In [ ]:
q_old_b7 = 0.4 # current critic prediction for the sampled state-action pair.
alpha_b7 = 0.5 # learning-rate fraction for a scalar illustration.
q_new_b7 = q_old_b7 + alpha_b7 * (y_b7 - q_old_b7) # move halfway toward the target.
print("Q old -> new:", round(q_old_b7, 3), "->", round(q_new_b7, 3)) # inspect the correction.
assert round(q_new_b7, 3) == 1.06 # verify the worked update.

In [ ]:
parts_b7 = [r_b7, gamma_b7 * q_next_b7, y_b7] # reuse the reward, discounted next-Q, and target.
labels_b7 = ["reward", "γ·Q_next", "target"] # label the Bellman components.
plt.figure(figsize=(4, 3)) # create a compact component chart.
plt.bar(labels_b7, parts_b7, color=["steelblue", "orange", "seagreen"]) # compare the terms visually.
plt.title("Basic 7: Bellman target pieces") # title the plot.
plt.ylabel("value") # label the vertical scale.
plt.ylim(0, max(parts_b7) * 1.2) # leave headroom above the target bar.
plt.show()

▶ What you'll see: the target bar equals the reward bar plus the discounted next-Q bar.

▶ What you'll see: the critic estimate moves toward the target without jumping all the way.

👀 Takeaway: bootstrapping lets the critic learn from incomplete returns, but the target depends on its own estimates.

### Basic 8 — Soft-update a target network

**Goal.** Apply `target ← τ online + (1−τ) target`, because DDPG and TD3 use slow target networks for stability. We build it in 2 steps.

In [ ]:
online_b8 = np.array([1.0, -0.5, 0.2]) # define online parameters after learning.
target_b8 = np.array([0.4, -0.1, 0.0]) # define older target parameters.
tau_b8 = 0.1 # choose a slow update rate.
new_target_b8 = tau_b8 * online_b8 + (1 - tau_b8) * target_b8 # compute the Polyak average.
print("new target:", np.round(new_target_b8, 3)) # inspect the slow-moving parameters.
assert np.allclose(np.round(new_target_b8, 3), [0.46, -0.14, 0.02]) # verify the update.

▶ What you'll see: the target parameters move only a small fraction toward the online parameters.

In [ ]:
plt.figure(figsize=(4, 3)) # create a parameter comparison chart.
plt.plot(online_b8, "o-", label="online") # draw online parameters.
plt.plot(target_b8, "o-", label="old target") # draw old target parameters.
plt.plot(new_target_b8, "o-", label="new target") # draw updated target parameters.
plt.title("Basic 8: soft target update") # title the plot.
plt.xlabel("parameter index") # label parameter index.
plt.ylabel("value") # label parameter value.
plt.legend() # show line labels.
plt.show() # display the plot.

▶ What you'll see: the new target line sits between the old target and online lines.

👀 Takeaway: target networks stabilize bootstrapping by changing slowly.

### Basic 9 — Take the smaller of twin critics

**Goal.** Compute `min(Q1,Q2)`, because TD3 reduces overestimated targets by clipping double Q values. We build it in 2 steps.

In [ ]:
q1_b9 = np.array([1.2, 0.7, 1.5, 0.4]) # define critic 1 estimates for candidate next actions.
q2_b9 = np.array([1.0, 0.9, 0.8, 0.5]) # define critic 2 estimates for the same candidates.
clipped_b9 = np.minimum(q1_b9, q2_b9) # TD3 target uses the smaller estimate.
print("clipped values:", clipped_b9) # inspect pessimistic values.
assert np.allclose(clipped_b9, [1.0, 0.7, 0.8, 0.4]) # verify elementwise minima.

▶ What you'll see: each candidate keeps whichever critic is less optimistic.

In [ ]:
plt.figure(figsize=(4, 3)) # create a twin-critic comparison.
x_b9 = np.arange(len(q1_b9)) # candidate indices.
plt.plot(x_b9, q1_b9, "o-", label="Q1") # plot critic 1.
plt.plot(x_b9, q2_b9, "o-", label="Q2") # plot critic 2.
plt.plot(x_b9, clipped_b9, "ko--", label="min") # plot the clipped target values.
plt.title("Basic 9: clipped double Q") # title the plot.
plt.xlabel("candidate") # label candidate axis.
plt.ylabel("Q estimate") # label value axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the minimum curve ignores the high-side critic whenever critics disagree.

👀 Takeaway: TD3 makes targets pessimistic to stop the actor from chasing one critic's lucky error.

### Basic 10 — Clip target policy smoothing noise

**Goal.** Add clipped noise to target actions, because TD3 smooths Bellman targets over a small action neighborhood. We build it in 2 steps.

In [ ]:
rng_b10 = np.random.default_rng(10) # create local reproducible randomness.
base_action_b10 = 0.7 # define the target actor's next action.
noise_b10 = np.clip(rng_b10.normal(0, 0.2, size=6), -0.3, 0.3) # draw and clip target smoothing noise.
smoothed_b10 = np.clip(base_action_b10 + noise_b10, -1, 1) # add noise and clip to action bounds.
print("noise:", np.round(noise_b10, 3)) # inspect clipped noise.
print("smoothed actions:", np.round(smoothed_b10, 3)) # inspect valid target actions.
assert np.all((smoothed_b10 >= -1) & (smoothed_b10 <= 1)) # verify action bounds.

▶ What you'll see: target actions are near 0.7 but not identical.

In [ ]:
plt.figure(figsize=(4, 3)) # create a small action plot.
plt.scatter(np.arange(len(smoothed_b10)), smoothed_b10, color="teal") # show each smoothed action.
plt.axhline(base_action_b10, color="black", linestyle="--", label="base target action") # mark the un-noised action.
plt.title("Basic 10: target smoothing noise") # title the plot.
plt.xlabel("sample") # label samples.
plt.ylabel("target action") # label action values.
plt.legend() # show base marker.
plt.show() # display the scatter plot.

▶ What you'll see: a small clipped neighborhood around the base target action.

👀 Takeaway: TD3 trains critics against locally smoothed targets rather than brittle single-action spikes.

## 🟡 Easy

### Easy 1 — Fit a quadratic critic with least squares

**Goal.** Learn a critic surface from sampled `(s,a,r)` data, because DDPG first needs a differentiable estimate of action value. We build it in 3 steps.

In [ ]:
rng_e1 = np.random.default_rng(11) # create reproducible training data.
s_e1 = rng_e1.uniform(-1, 1, size=200) # sample states.
a_e1 = rng_e1.uniform(-1.5, 1.5, size=200) # sample exploratory continuous actions.
y_e1 = -((a_e1 - s_e1) ** 2) # compute the true one-step target values.
print("samples:", s_e1.shape[0]) # inspect data size.

▶ What you'll see: 200 off-policy state-action samples for critic fitting.

In [ ]:
X_e1 = np.column_stack([np.ones_like(s_e1), s_e1, a_e1, s_e1**2, s_e1*a_e1, a_e1**2]) # build quadratic critic features.
w_e1 = np.linalg.lstsq(X_e1, y_e1, rcond=None)[0] # solve least squares with NumPy.
print("weights:", np.round(w_e1, 3)) # inspect learned critic parameters.
assert np.allclose(np.round(w_e1, 3), [0, 0, 0, -1, 2, -1]) # verify exact recovery of the quadratic reward.

In [ ]:
pred_e1 = X_e1 @ w_e1 # compute fitted critic values on training samples.
rmse_e1 = float(np.sqrt(np.mean((pred_e1 - y_e1) ** 2))) # measure fit error.
print("critic RMSE:", round(rmse_e1, 6)) # inspect near-zero error.
plt.figure(figsize=(4, 3)) # create a predicted-vs-true plot.
plt.scatter(y_e1, pred_e1, s=12, alpha=0.6, color="teal") # compare values.
plt.plot([y_e1.min(), y_e1.max()], [y_e1.min(), y_e1.max()], "k--") # ideal diagonal.
plt.title("Easy 1: fitted critic") # title the plot.
plt.xlabel("true Q") # label true values.
plt.ylabel("predicted Q") # label predicted values.
plt.show() # display the scatter.

▶ What you'll see: points lie on the diagonal because the feature basis matches the true quadratic.

👀 Takeaway: a critic is a supervised regression model whose labels are rewards or Bellman targets.

### Easy 2 — Improve an actor with critic gradients

**Goal.** Train `μθ(s)=θs` using deterministic policy gradients, because the actor should climb the critic surface. We build it in 4 steps.

In [ ]:
states_e2 = np.linspace(-1, 1, 41) # create states used to estimate the policy objective.
theta_e2 = 0.0 # start with an actor that always outputs zero.
lr_e2 = 0.15 # choose a stable actor learning rate.
trace_e2 = [] # store θ values over updates.
print("initial theta:", theta_e2) # inspect the starting policy.

▶ What you'll see: the actor starts with no response to state.

In [ ]:
for step_e2 in range(30): # run repeated deterministic policy-gradient steps.
    actions_e2 = theta_e2 * states_e2 # compute actor actions.
    dq_da_e2 = -2 * (actions_e2 - states_e2) # critic action gradient for Q=-(a-s)^2.
    grad_e2 = float(np.mean(dq_da_e2 * states_e2)) # chain rule: dμ/dθ=s.
    theta_e2 += lr_e2 * grad_e2 # ascend the objective.
    trace_e2.append(theta_e2) # record the updated actor parameter.
print("final theta:", round(theta_e2, 3)) # inspect learned controller gain.
assert theta_e2 > 0.95 # verify the actor moved close to the optimal gain 1.

In [ ]:
obj_e2 = [float(np.mean(-(((t * states_e2) - states_e2) ** 2))) for t in trace_e2] # compute objective along training.
print("objective start -> end:", round(obj_e2[0], 3), "->", round(obj_e2[-1], 3)) # inspect improvement.

In [ ]:
plt.figure(figsize=(5, 3)) # create an actor-training curve.
plt.plot(trace_e2, marker="o", color="purple") # plot θ over updates.
plt.axhline(1.0, color="black", linestyle="--", label="optimal θ=1") # mark the optimum.
plt.title("Easy 2: actor learns from critic gradients") # title the curve.
plt.xlabel("actor update") # label updates.
plt.ylabel("θ") # label actor parameter.
plt.legend() # show optimum label.
plt.show() # display the curve.

▶ What you'll see: θ rises smoothly toward 1, the optimal deterministic controller.

👀 Takeaway: deterministic policy gradient is gradient ascent on critic-estimated value.

### Easy 3 — Compute DDPG critic targets from replay

**Goal.** Build Bellman targets for a mini-batch, because DDPG trains the critic from replayed transitions. We build it in 3 steps.

In [ ]:
rng_e3 = np.random.default_rng(13) # create reproducible replay samples.
s_e3 = rng_e3.uniform(-1, 1, size=12) # sample current states.
a_e3 = np.clip(s_e3 + rng_e3.normal(0, 0.3, size=12), -1, 1) # sample exploratory actions.
sp_e3 = 0.7 * s_e3 + 0.3 * a_e3 # define simple next states.
r_e3 = -((a_e3 - s_e3) ** 2) - 0.05 * sp_e3**2 # define rewards.
print("batch size:", len(s_e3)) # inspect mini-batch size.

▶ What you'll see: a mini-batch of transitions that could have come from a replay buffer.

In [ ]:
gamma_e3 = 0.9 # discount future critic values.
theta_targ_e3 = 0.8 # define target actor gain.
next_a_e3 = theta_targ_e3 * sp_e3 # compute target actor actions at next states.
q_next_e3 = -((next_a_e3 - sp_e3) ** 2) # evaluate target critic for the toy environment.
y_e3 = r_e3 + gamma_e3 * q_next_e3 # compute DDPG Bellman targets.
print("first 4 targets:", np.round(y_e3[:4], 3)) # inspect target values.
assert y_e3.shape == r_e3.shape # verify one target per transition.

In [ ]:
plt.figure(figsize=(4, 3)) # create a target diagnostic plot.
plt.scatter(r_e3, y_e3, color="teal") # compare immediate reward with bootstrapped target.
plt.title("Easy 3: reward vs Bellman target") # title the plot.
plt.xlabel("immediate reward r") # label immediate rewards.
plt.ylabel("target y") # label targets.
plt.show() # display the scatter.

▶ What you'll see: targets are usually below immediate rewards here because future squared-error penalties are negative.

👀 Takeaway: the critic target is not just reward; it includes discounted next-state value from target networks.

### Easy 4 — Compare DDPG and TD3 targets

**Goal.** Replace a single target critic with `min(Q1,Q2)`, because TD3 clips optimistic target values. We build it in 3 steps.

In [ ]:
q1_next_e4 = np.array([0.9, 1.4, 0.3, 1.1, 0.7]) # define target critic 1 estimates.
q2_next_e4 = np.array([0.8, 0.6, 0.4, 0.9, 0.75]) # define target critic 2 estimates.
r_e4 = np.array([0.2, 0.1, -0.1, 0.0, 0.3]) # define immediate rewards.
gamma_e4 = 0.9 # define discount.
print("Q1:", q1_next_e4) # inspect critic 1.
print("Q2:", q2_next_e4) # inspect critic 2.

▶ What you'll see: critic 1 is especially optimistic on the second transition.

In [ ]:
y_ddpg_e4 = r_e4 + gamma_e4 * q1_next_e4 # compute single-critic DDPG-style targets.
y_td3_e4 = r_e4 + gamma_e4 * np.minimum(q1_next_e4, q2_next_e4) # compute clipped double-Q TD3 targets.
print("DDPG targets:", np.round(y_ddpg_e4, 3)) # inspect optimistic targets.
print("TD3 targets:", np.round(y_td3_e4, 3)) # inspect clipped targets.
assert y_td3_e4[1] < y_ddpg_e4[1] # verify clipping lowers the optimistic second target.

In [ ]:
plt.figure(figsize=(5, 3)) # create a target comparison chart.
x_e4 = np.arange(len(r_e4)) # transition indices.
plt.plot(x_e4, y_ddpg_e4, "o-", label="DDPG target") # plot single-critic targets.
plt.plot(x_e4, y_td3_e4, "o-", label="TD3 target") # plot clipped targets.
plt.title("Easy 4: clipped targets are lower when critics disagree") # title the chart.
plt.xlabel("transition") # label transitions.
plt.ylabel("Bellman target") # label target values.
plt.legend() # show target labels.
plt.show() # display the chart.

▶ What you'll see: TD3 matches or lowers the DDPG target, especially where one critic is high.

👀 Takeaway: clipped double-Q directly attacks overestimation in the Bellman target.

### Easy 5 — Smooth target actions before evaluating Q

**Goal.** Average a spiky target critic over small action noise, because TD3 target smoothing punishes brittle peaks. We build it in 3 steps.

In [ ]:
rng_e5 = np.random.default_rng(15) # create reproducible smoothing noise.
base_e5 = 0.55 # define the target actor's action.
noise_e5 = np.clip(rng_e5.normal(0, 0.18, size=1000), -0.25, 0.25) # draw clipped target noise.
a_smooth_e5 = np.clip(base_e5 + noise_e5, -1, 1) # add noise and enforce action bounds.
print("action mean/std:", round(float(a_smooth_e5.mean()), 3), round(float(a_smooth_e5.std()), 3)) # inspect smoothed action distribution.

▶ What you'll see: smoothed target actions remain near the base action.

In [ ]:
def q_spike_e5(a): # define a critic with a narrow optimistic spike.
    return -((a - 0.35) ** 2) + 0.5 * np.exp(-((a - 0.55) / 0.03) ** 2)
point_e5 = float(q_spike_e5(base_e5)) # evaluate Q at the exact target action.
avg_e5 = float(np.mean(q_spike_e5(a_smooth_e5))) # evaluate averaged noisy target Q.
print("point Q:", round(point_e5, 3), "smoothed Q:", round(avg_e5, 3)) # inspect smoothing effect.
assert avg_e5 < point_e5 # verify the narrow spike is damped.

In [ ]:
a_line_e5 = np.linspace(0, 0.9, 200) # create an action line around the spike.
plt.figure(figsize=(5, 3)) # create a smoothing plot.
plt.plot(a_line_e5, q_spike_e5(a_line_e5), color="purple", label="critic") # draw the spiky critic.
plt.hist(a_smooth_e5, bins=25, density=True, alpha=0.25, color="gray", label="target-noise density") # overlay action noise density.
plt.axvline(base_e5, color="red", linestyle="--", label="base action") # mark the base action.
plt.title("Easy 5: target policy smoothing") # title the plot.
plt.xlabel("action") # label actions.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: the narrow spike is high at one action but covers little of the smoothing neighborhood.

👀 Takeaway: TD3 trains on locally robust target values instead of single-point critic artifacts.

## 🔴 Advanced

### Advanced 1 — Run a tiny DDPG training loop

**Goal.** Alternate critic fitting and actor-gradient updates, because DDPG learns a value surface and a deterministic policy together. We build it in 5 steps.

In [ ]:
rng_a1 = np.random.default_rng(21) # create reproducible replay data.
s_a1 = rng_a1.uniform(-1, 1, size=300) # sample states from replay.
a_a1 = rng_a1.uniform(-1.2, 1.2, size=300) # sample exploratory actions from replay.
y_a1 = -((a_a1 - s_a1) ** 2) # one-step critic labels for the toy environment.
print("replay samples:", len(s_a1)) # inspect replay size.

▶ What you'll see: the replay buffer spans many states and actions.

In [ ]:
X_a1 = np.column_stack([np.ones_like(s_a1), s_a1, a_a1, s_a1**2, s_a1*a_a1, a_a1**2]) # build critic features.
w_a1 = np.linalg.lstsq(X_a1, y_a1, rcond=None)[0] # fit the critic by least squares.
theta_a1 = -0.4 # initialize a bad actor with the wrong sign.
states_a1 = np.linspace(-1, 1, 101) # states used for actor updates.
trace_a1 = [] # store actor parameter history.
print("critic weights:", np.round(w_a1, 3)) # inspect critic fit.

In [ ]:
for step_a1 in range(45): # run actor updates against the fixed fitted critic.
    act_a1 = theta_a1 * states_a1 # compute actor actions.
    dq_da_a1 = w_a1[2] + w_a1[4] * states_a1 + 2 * w_a1[5] * act_a1 # differentiate quadratic critic wrt action.
    grad_a1 = float(np.mean(dq_da_a1 * states_a1)) # deterministic policy gradient for θ.
    theta_a1 += 0.12 * grad_a1 # update actor by gradient ascent.
    trace_a1.append(theta_a1) # record θ.
print("theta final:", round(theta_a1, 3)) # inspect final actor.
assert theta_a1 > 0.9 # verify it recovered the near-optimal controller.

In [ ]:
returns_a1 = [float(np.mean(-(((t * states_a1) - states_a1) ** 2))) for t in trace_a1] # compute policy quality over training.
print("return start -> end:", round(returns_a1[0], 3), "->", round(returns_a1[-1], 3)) # inspect improvement.
assert returns_a1[-1] > returns_a1[0] # verify learning improved the policy.

In [ ]:
plt.figure(figsize=(5, 3)) # create a training curve.
plt.plot(trace_a1, label="actor θ", color="teal") # plot actor gain.
plt.axhline(1.0, color="black", linestyle="--", label="optimal") # mark optimal gain.
plt.title("Advanced 1: tiny DDPG actor improvement") # title the curve.
plt.xlabel("actor update") # label updates.
plt.ylabel("θ") # label parameter.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: even from the wrong sign, the actor follows the fitted critic toward the optimal controller.

👀 Takeaway: DDPG is critic regression plus deterministic actor gradient ascent, connected by `∂Q/∂a`.

### Advanced 2 — Show overestimation from critic noise

**Goal.** Compare a noisy single critic against twin critics, because deterministic actors exploit optimistic errors. We build it in 4 steps.

In [ ]:
rng_a2 = np.random.default_rng(22) # create reproducible critic noise.
a_grid_a2 = np.linspace(-1, 1, 201) # candidate actions for visualization.
true_a2 = -((a_grid_a2 - 0.2) ** 2) # true Q curve with optimum at 0.2.
noise1_a2 = 0.05 * np.sin(13 * a_grid_a2) + 0.5 * np.exp(-((a_grid_a2 - 0.75) / 0.08) ** 2) # critic 1 optimistic artifact.
noise2_a2 = 0.08 * np.cos(9 * a_grid_a2) - 0.04 * np.exp(-((a_grid_a2 - 0.75) / 0.1) ** 2) # different critic 2 error.
q1_a2 = true_a2 + noise1_a2 # critic 1 estimate.
q2_a2 = true_a2 + noise2_a2 # critic 2 estimate.
print("true optimal action:", 0.2) # inspect the analytic optimum.

▶ What you'll see: two critics share the same true bowl but have different errors.

In [ ]:
best_q1_a2 = float(a_grid_a2[np.argmax(q1_a2)]) # action a single actor would pick from critic 1.
best_min_a2 = float(a_grid_a2[np.argmax(np.minimum(q1_a2, q2_a2))]) # action selected by clipped double-Q.
print("best Q1 action:", round(best_q1_a2, 3)) # inspect optimistic action.
print("best min action:", round(best_min_a2, 3)) # inspect clipped action.
assert abs(best_min_a2 - 0.2) < abs(best_q1_a2 - 0.2) # verify clipped double-Q is closer to the true optimum.

In [ ]:
gap_a2 = float(np.max(q1_a2) - true_a2[np.argmax(q1_a2)]) # estimate overoptimism at selected Q1 action.
print("overestimation at Q1-selected action:", round(gap_a2, 3)) # inspect high-side error.

In [ ]:
plt.figure(figsize=(5, 3)) # create overestimation plot.
plt.plot(a_grid_a2, true_a2, "k--", label="true Q") # plot true value.
plt.plot(a_grid_a2, q1_a2, label="Q1 noisy") # plot critic 1.
plt.plot(a_grid_a2, np.minimum(q1_a2, q2_a2), label="min(Q1,Q2)") # plot clipped estimate.
plt.axvline(best_q1_a2, color="red", linestyle=":", label="Q1 argmax") # mark Q1 choice.
plt.axvline(best_min_a2, color="green", linestyle=":", label="min argmax") # mark clipped choice.
plt.title("Advanced 2: overestimation and clipped double-Q") # title the plot.
plt.xlabel("action") # label action axis.
plt.ylabel("Q") # label value axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: a single noisy critic can move the actor toward an artificial high-value bump.

👀 Takeaway: TD3's twin critics are a guardrail against actor-driven exploitation of critic noise.

### Advanced 3 — Delay actor updates while critics settle

**Goal.** Simulate critic error and actor updates on different schedules, because TD3 updates the policy less often than the value functions. We build it in 4 steps.

In [ ]:
steps_a3 = np.arange(1, 31) # define training steps.
critic_error_a3 = np.exp(-steps_a3 / 8) # model critic error shrinking as critics train.
actor_every_a3 = 2 # TD3 commonly delays policy updates.
actor_flags_a3 = (steps_a3 % actor_every_a3 == 0) # mark actor update steps.
print("actor updates:", int(actor_flags_a3.sum()), "critic updates:", len(steps_a3)) # inspect schedules.

▶ What you'll see: critics update every step; actor updates happen half as often.

In [ ]:
theta_a3 = 0.0 # initialize actor parameter.
theta_trace_a3 = [] # store parameter values.
for idx_a3, err_a3 in enumerate(critic_error_a3): # step through training.
    if actor_flags_a3[idx_a3]: # update actor only on delayed schedule.
        noisy_grad_a3 = (1.0 - theta_a3) + 0.4 * err_a3 # use a biased gradient that improves as critic error shrinks.
        theta_a3 += 0.2 * noisy_grad_a3 # apply actor update.
    theta_trace_a3.append(theta_a3) # record current actor.
print("final theta:", round(theta_a3, 3)) # inspect delayed actor result.
assert theta_a3 > 0.8 # verify the actor still learns.

In [ ]:
plt.figure(figsize=(5, 3)) # create a two-axis-style diagnostic.
plt.plot(steps_a3, critic_error_a3, label="critic error proxy", color="red") # plot critic error shrinkage.
plt.step(steps_a3, theta_trace_a3, where="mid", label="actor θ", color="teal") # plot actor parameter changes.
plt.title("Advanced 3: delayed actor updates") # title the plot.
plt.xlabel("training step") # label steps.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: actor changes are stair-stepped, giving critic error time to fall between policy moves.

In [ ]:
update_steps_a3 = steps_a3[actor_flags_a3] # collect actual actor update steps.
print("actor updated on steps:", update_steps_a3[:8], "...") # inspect the delayed schedule.

▶ What you'll see: only even-numbered steps update the policy.

👀 Takeaway: delaying actor updates reduces harmful feedback from immature critic estimates.

### Advanced 4 — Compare point targets with smoothed targets

**Goal.** Quantify how target smoothing reduces narrow critic spikes, because TD3 wants robust value estimates around the target action. We build it in 4 steps.

In [ ]:
rng_a4 = np.random.default_rng(24) # create reproducible smoothing samples.
base_a4 = 0.5 # choose target action at a spike.
noise_a4 = np.clip(rng_a4.normal(0, 0.2, size=4000), -0.3, 0.3) # draw clipped smoothing noise.
a_noisy_a4 = np.clip(base_a4 + noise_a4, -1, 1) # apply action bounds.
print("noisy action std:", round(float(a_noisy_a4.std()), 3)) # inspect smoothing spread.

▶ What you'll see: target actions cover a local neighborhood around the actor action.

In [ ]:
def q_narrow_a4(a): # define a critic with a narrow high spike and a broad useful bowl.
    return -((a - 0.25) ** 2) + 0.8 * np.exp(-((a - 0.5) / 0.025) ** 2)
point_a4 = float(q_narrow_a4(base_a4)) # evaluate at the exact target action.
smooth_a4 = float(np.mean(q_narrow_a4(a_noisy_a4))) # average over noisy target actions.
print("point target:", round(point_a4, 3), "smoothed target:", round(smooth_a4, 3)) # inspect spike reduction.
assert smooth_a4 < 0.5 * point_a4 # verify smoothing strongly damps the narrow spike.

In [ ]:
percentile_a4 = np.percentile(q_narrow_a4(a_noisy_a4), [10, 50, 90]) # summarize target-value distribution.
print("smoothed Q percentiles:", np.round(percentile_a4, 3)) # inspect how rarely the spike appears.

In [ ]:
a_line_a4 = np.linspace(0.0, 0.8, 250) # create line for critic visualization.
plt.figure(figsize=(5, 3)) # create smoothing diagnostic plot.
plt.plot(a_line_a4, q_narrow_a4(a_line_a4), color="purple", label="target critic") # plot critic values.
plt.axvline(base_a4, color="red", linestyle="--", label="point action") # mark base action.
plt.axhline(smooth_a4, color="green", linestyle="--", label="mean smoothed Q") # mark smoothed average.
plt.title("Advanced 4: smoothing damps narrow spikes") # title the plot.
plt.xlabel("action") # label action axis.
plt.ylabel("Q") # label value axis.
plt.legend() # show labels.
plt.show() # display the plot.

▶ What you'll see: a tall narrow spike at the point action but a much lower neighborhood average.

👀 Takeaway: target smoothing makes the Bellman target care about local robustness, not just one fragile action.

### Advanced 5 — Assemble one TD3 update from scratch

**Goal.** Put replay, target smoothing, twin critics, and delayed actor logic into one small update, because TD3 is DDPG plus these stabilizers. We build it in 5 steps.

In [ ]:
rng_a5 = np.random.default_rng(25) # create reproducible mini-batch.
s_a5 = rng_a5.uniform(-1, 1, size=10) # sample current states.
a_a5 = rng_a5.uniform(-1, 1, size=10) # sample replay actions.
sp_a5 = 0.6 * s_a5 + 0.4 * a_a5 # define next states.
r_a5 = -((a_a5 - s_a5) ** 2) # define immediate rewards.
print("mini-batch size:", len(s_a5)) # inspect batch size.

▶ What you'll see: a small replay mini-batch ready for a TD3-style target.

In [ ]:
theta_target_a5 = 0.7 # define target actor gain.
noise_a5 = np.clip(rng_a5.normal(0, 0.2, size=len(sp_a5)), -0.3, 0.3) # draw clipped target smoothing noise.
next_a_a5 = np.clip(theta_target_a5 * sp_a5 + noise_a5, -1, 1) # compute smoothed target actions.
print("first smoothed actions:", np.round(next_a_a5[:4], 3)) # inspect target actions.
assert np.all((next_a_a5 >= -1) & (next_a_a5 <= 1)) # verify valid action range.

In [ ]:
q1_next_a5 = -((next_a_a5 - sp_a5) ** 2) + 0.08 * np.sin(7 * next_a_a5) # target critic 1 with small error.
q2_next_a5 = -((next_a_a5 - sp_a5) ** 2) - 0.04 * np.cos(5 * next_a_a5) # target critic 2 with different error.
y_td3_a5 = r_a5 + 0.9 * np.minimum(q1_next_a5, q2_next_a5) # compute clipped double-Q target.
print("first TD3 targets:", np.round(y_td3_a5[:4], 3)) # inspect targets.
assert y_td3_a5.shape == r_a5.shape # verify one target per transition.

In [ ]:
policy_delay_a5 = 2 # update actor every two critic steps.
critic_step_a5 = 6 # choose a critic step index.
should_update_actor_a5 = (critic_step_a5 % policy_delay_a5 == 0) # decide whether actor updates now.
print("update actor now?", should_update_actor_a5) # inspect delayed-policy rule.
assert should_update_actor_a5 # verify step 6 triggers an actor update.

In [ ]:
plt.figure(figsize=(5, 3)) # create target diagnostic figure.
plt.plot(y_td3_a5, "o-", color="teal", label="TD3 target") # plot final targets.
plt.plot(r_a5, "o--", color="gray", label="immediate reward") # plot immediate rewards.
plt.title("Advanced 5: one TD3 target batch") # title the plot.
plt.xlabel("transition") # label transitions.
plt.ylabel("value") # label values.
plt.legend() # show labels.
plt.show() # display the chart.

▶ What you'll see: TD3 targets combine reward, smoothed target actions, and the smaller of two target critics.

👀 Takeaway: TD3 keeps DDPG's deterministic actor-critic core but makes each bootstrapped target harder to overexploit.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

DDPG and TD3 learn smooth actions by differentiating through a critic.

Deterministic actor-critic methods replace discrete action tables with continuous actions. TD3 adds twin critics, delayed policy updates, and target smoothing to control over-estimated bootstrapped targets. Save a copy to Drive to edit.

In [ ]:

import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 1113
rng = np.random.default_rng(SEED)
random.seed(SEED)

ACTIONS = np.array([
    [-1, 0],
    [1, 0],
    [0, -1],
    [0, 1],
])
ACTION_NAMES = np.array(["U", "D", "L", "R"])

@dataclass
class LadderEnv:
    name: str
    height: int
    width: int
    start: tuple
    goal: tuple
    walls: tuple
    slip: float
    wind: float
    step_cost: float
    goal_reward: float
    traps: dict
    max_steps: int
    continuous: bool = False

    @property
    def n_states(self):
        return self.height * self.width

    @property
    def n_actions(self):
        return len(ACTIONS)

    def state_index(self, pos):
        row, col = pos
        return row * self.width + col

    def index_state(self, idx):
        row = idx // self.width
        col = idx % self.width
        return (row, col)

    def reset(self):
        return self.state_index(self.start)

    def move(self, pos, action, local_rng):
        actual = int(action)
        if local_rng.random() < self.slip:
            actual = int(local_rng.integers(0, self.n_actions))
        delta = ACTIONS[actual].copy()
        if self.wind > 0.0 and local_rng.random() < self.wind:
            delta = delta + np.array([-1, 0])
        nxt = (pos[0] + int(delta[0]), pos[1] + int(delta[1]))
        bad_row = nxt[0] < 0 or nxt[0] >= self.height
        bad_col = nxt[1] < 0 or nxt[1] >= self.width
        if bad_row or bad_col or nxt in self.walls:
            nxt = pos
        return nxt

    def step(self, state, action, local_rng):
        pos = self.index_state(int(state))
        nxt = self.move(pos, action, local_rng)
        reward = self.step_cost
        done = False
        if nxt in self.traps:
            reward = reward + float(self.traps[nxt])
        if nxt == self.goal:
            reward = reward + self.goal_reward
            done = True
        return self.state_index(nxt), reward, done


def make_rl_ladder(continuous=False):
    envs = []
    envs.append(LadderEnv("D1 two-state chain", 1, 2, (0, 0), (0, 1), tuple(), 0.0, 0.0, 0.0, 1.0, {}, 4, continuous))
    envs.append(LadderEnv("D2 slippery 3-state", 1, 3, (0, 0), (0, 2), tuple(), 0.15, 0.0, -0.01, 1.0, {}, 8, continuous))
    envs.append(LadderEnv("D3 4x4 gridworld", 4, 4, (3, 0), (0, 3), ((1, 1),), 0.05, 0.0, -0.02, 1.0, {(2, 2): -0.25}, 24, continuous))
    envs.append(LadderEnv("D4 windy stochastic grid", 5, 5, (4, 0), (0, 4), ((1, 1), (2, 1), (3, 3)), 0.12, 0.18, -0.025, 1.1, {(2, 3): -0.4}, 35, continuous))
    envs.append(LadderEnv("D5 sparse reward grid", 6, 6, (5, 0), (0, 5), ((1, 1), (1, 2), (2, 2), (3, 4), (4, 1)), 0.18, 0.20, -0.03, 1.5, {(2, 4): -0.6, (4, 4): -0.3}, 50, continuous))
    return envs


def softmax(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)


def discounted_returns(rewards, gamma):
    returns = np.zeros(len(rewards), dtype=float)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = float(rewards[t]) + gamma * running
        returns[t] = running
    return returns


def rollout(env, logits, local_rng, gamma=0.9):
    states = []
    actions = []
    rewards = []
    state = env.reset()
    for _ in range(env.max_steps):
        probs = softmax(logits[state])
        action = int(local_rng.choice(env.n_actions, p=probs))
        next_state, reward, done = env.step(state, action, local_rng)
        states.append(state)
        actions.append(action)
        rewards.append(reward)
        state = next_state
        if done:
            break
    returns = discounted_returns(np.array(rewards), gamma)
    return np.array(states), np.array(actions), np.array(rewards), returns


def evaluate_policy(env, logits, episodes=20, gamma=0.9):
    values = []
    local_rng = np.random.default_rng(SEED + env.n_states)
    for _ in range(episodes):
        _, _, rewards, _ = rollout(env, logits, local_rng, gamma)
        values.append(float(np.sum(rewards)))
    return float(np.mean(values))


def value_iteration(env, gamma=0.9, iterations=120):
    values = np.zeros(env.n_states)
    local_rng = np.random.default_rng(SEED)
    for _ in range(iterations):
        new_values = values.copy()
        for state in range(env.n_states):
            pos = env.index_state(state)
            if pos == env.goal:
                continue
            q_values = []
            for action in range(env.n_actions):
                next_state, reward, done = env.step(state, action, local_rng)
                q_values.append(reward + gamma * values[next_state] * (1.0 - float(done)))
            new_values[state] = np.max(q_values)
        values = new_values
    return values


def greedy_logits_from_values(env, values, gamma=0.9, scale=5.0):
    logits = np.zeros((env.n_states, env.n_actions))
    local_rng = np.random.default_rng(SEED + 7)
    for state in range(env.n_states):
        for action in range(env.n_actions):
            next_state, reward, done = env.step(state, action, local_rng)
            logits[state, action] = scale * (reward + gamma * values[next_state] * (1.0 - float(done)))
    return logits


def plot_policy_panel(ax, env, values, logits, title):
    grid = values.reshape(env.height, env.width)
    ax.imshow(grid, cmap="viridis")
    probs = softmax(logits)
    for state in range(env.n_states):
        row, col = env.index_state(state)
        if (row, col) in env.walls:
            ax.text(col, row, "#", ha="center", va="center", color="white")
            continue
        best = int(np.argmax(probs[state]))
        ax.text(col, row, ACTION_NAMES[best], ha="center", va="center", color="white")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def summarize_ladder(envs):
    for env in envs:
        print(env.name, "states", env.n_states, "actions", env.n_actions, "slip", env.slip, "wind", env.wind)
        print("start", env.start, "goal", env.goal, "walls", len(env.walls), "traps", env.traps)


def train_reinforce(env, episodes=80, gamma=0.9, lr=0.08, baseline=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        advantages = returns.copy()
        if baseline:
            advantages = returns - value[states]
            for state, target in zip(states, returns):
                value[state] = value[state] + 0.15 * (target - value[state])
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + lr * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def train_actor_critic(env, episodes=80, gamma=0.9, actor_lr=0.05, critic_lr=0.12, normalize=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + 2 * env.n_states)
    for _ in range(episodes):
        states = []
        actions = []
        deltas = []
        rewards = []
        state = env.reset()
        for _ in range(env.max_steps):
            probs = softmax(logits[state])
            action = int(local_rng.choice(env.n_actions, p=probs))
            next_state, reward, done = env.step(state, action, local_rng)
            target = reward + gamma * value[next_state] * (1.0 - float(done))
            delta = target - value[state]
            value[state] = value[state] + critic_lr * delta
            states.append(state)
            actions.append(action)
            deltas.append(delta)
            rewards.append(reward)
            state = next_state
            if done:
                break
        advantages = np.array(deltas)
        if normalize and len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + actor_lr * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def gae_advantages(rewards, values, next_values, dones, gamma=0.9, lam=0.95):
    deltas = rewards + gamma * next_values * (1.0 - dones) - values
    adv = np.zeros_like(rewards, dtype=float)
    running = 0.0
    for t in range(len(rewards) - 1, -1, -1):
        running = deltas[t] + gamma * lam * (1.0 - dones[t]) * running
        adv[t] = running
    return deltas, adv


def train_gae_policy(env, lam=0.95, episodes=80, gamma=0.9):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + int(100 * lam) + env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        next_values = np.zeros(len(states))
        dones = np.zeros(len(states))
        for i, state in enumerate(states):
            if i + 1 < len(states):
                next_values[i] = value[states[i + 1]]
            else:
                dones[i] = 1.0
        _, advantages = gae_advantages(rewards, value[states], next_values, dones, gamma, lam)
        for state, target in zip(states, returns):
            value[state] = value[state] + 0.12 * (target - value[state])
        if len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for state, action, advantage in zip(states, actions, advantages):
            probs = softmax(logits[state])
            grad = -probs
            grad[action] = grad[action] + 1.0
            logits[state] = logits[state] + 0.05 * advantage * grad
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def ppo_surrogate(old_probs, new_probs, actions, advantages, clip=0.2):
    chosen_old = old_probs[np.arange(len(actions)), actions]
    chosen_new = new_probs[np.arange(len(actions)), actions]
    ratios = chosen_new / np.maximum(chosen_old, 1e-8)
    unclipped = ratios * advantages
    clipped = np.clip(ratios, 1.0 - clip, 1.0 + clip) * advantages
    return ratios, np.minimum(unclipped, clipped)


def train_ppo(env, episodes=80, gamma=0.9, clip=0.2, clipped=True):
    logits = np.zeros((env.n_states, env.n_actions))
    value = np.zeros(env.n_states)
    curve = []
    local_rng = np.random.default_rng(SEED + 3 * env.n_states)
    for _ in range(episodes):
        states, actions, rewards, returns = rollout(env, logits, local_rng, gamma)
        if len(states) == 0:
            continue
        old_probs = softmax(logits[states])
        advantages = returns - value[states]
        if len(advantages) > 1:
            advantages = (advantages - np.mean(advantages)) / (np.std(advantages) + 1e-8)
        for _epoch in range(3):
            new_probs = softmax(logits[states])
            ratios, weights = ppo_surrogate(old_probs, new_probs, actions, advantages, clip)
            if not clipped:
                weights = ratios * advantages
            for state, action, weight in zip(states, actions, weights):
                probs = softmax(logits[state])
                grad = -probs
                grad[action] = grad[action] + 1.0
                logits[state] = logits[state] + 0.04 * weight * grad
        for state, target in zip(states, returns):
            value[state] = value[state] + 0.15 * (target - value[state])
        curve.append(float(np.sum(rewards)))
    return logits, value, np.array(curve)


def distributional_backup(rewards, gamma):
    atoms = discounted_returns(np.array(rewards, dtype=float), gamma)
    probs = np.ones_like(atoms) / len(atoms)
    return atoms, probs, float(atoms[0])


def train_distributional(env, atoms=21, episodes=60, gamma=0.9):
    support = np.linspace(-1.0, 2.0, atoms)
    pmf = np.ones((env.n_states, env.n_actions, atoms)) / atoms
    logits = np.zeros((env.n_states, env.n_actions))
    curve = []
    errors = []
    optimal = value_iteration(env, gamma)
    local_rng = np.random.default_rng(SEED + 4 * env.n_states)
    for _ in range(episodes):
        state = env.reset()
        episode_reward = 0.0
        for _step in range(env.max_steps):
            means = np.sum(pmf[state] * support[None, :], axis=1)
            action = int(np.argmax(means + local_rng.normal(0.0, 0.03, env.n_actions)))
            next_state, reward, done = env.step(state, action, local_rng)
            next_action = int(np.argmax(np.sum(pmf[next_state] * support[None, :], axis=1)))
            shifted = reward + gamma * support * (1.0 - float(done))
            target = np.interp(support, shifted, pmf[next_state, next_action], left=0.0, right=0.0)
            if np.sum(target) <= 0.0:
                nearest = int(np.argmin(np.abs(support - reward)))
                target = np.zeros(atoms)
                target[nearest] = 1.0
            target = target / np.sum(target)
            pmf[state, action] = 0.9 * pmf[state, action] + 0.1 * target
            episode_reward = episode_reward + reward
            state = next_state
            if done:
                break
        learned = np.max(np.sum(pmf * support[None, None, :], axis=2), axis=1)
        errors.append(float(np.mean(np.abs(learned - optimal))))
        curve.append(episode_reward)
    values = np.max(np.sum(pmf * support[None, None, :], axis=2), axis=1)
    logits = greedy_logits_from_values(env, values, gamma)
    spread = np.mean(np.std(pmf * support[None, None, :], axis=2))
    return logits, values, np.array(errors), float(spread)


def continuous_features(state, action):
    return np.array([1.0, state, action, state * action, action * action])


def deterministic_actor_critic(reward, next_q, gamma):
    target = reward + gamma * next_q
    critic_prediction = 0.4
    critic_error = target - critic_prediction
    return target, critic_error


def td3_toy_update(state, reward, next_state, actor_w, q1_w, q2_w, gamma=0.9, noise=0.05):
    action = float(np.tanh(actor_w * state))
    next_action = float(np.clip(np.tanh(actor_w * next_state) + noise, -1.0, 1.0))
    q1_next = float(continuous_features(next_state, next_action) @ q1_w)
    q2_next = float(continuous_features(next_state, next_action) @ q2_w)
    target = reward + gamma * min(q1_next, q2_next)
    prediction = float(continuous_features(state, action) @ q1_w)
    td_error = target - prediction
    q1_w = q1_w + 0.05 * td_error * continuous_features(state, action)
    return action, target, td_error, q1_w


def train_td3_ladder(env, episodes=70, gamma=0.9, twin=True):
    actor_w = 0.2
    q1_w = np.array([0.0, 0.2, 0.1, 0.0, -0.05])
    q2_w = np.array([-0.02, 0.15, 0.08, 0.0, -0.08])
    curve = []
    local_rng = np.random.default_rng(SEED + 5 * env.n_states)
    for _ in range(episodes):
        state = 0.0
        total = 0.0
        for _step in range(env.max_steps):
            action = float(np.clip(np.tanh(actor_w * state) + local_rng.normal(0.0, 0.15), -1.0, 1.0))
            target_position = 1.0
            next_state = float(np.clip(state + 0.25 * action + local_rng.normal(0.0, 0.03), -1.2, 1.2))
            reward = -abs(target_position - next_state) - 0.05 * action * action
            next_action = float(np.clip(np.tanh(actor_w * next_state) + local_rng.normal(0.0, 0.05), -1.0, 1.0))
            q1_next = float(continuous_features(next_state, next_action) @ q1_w)
            q2_next = float(continuous_features(next_state, next_action) @ q2_w)
            next_value = min(q1_next, q2_next) if twin else q1_next
            target = reward + gamma * next_value
            feat = continuous_features(state, action)
            td_error_1 = target - float(feat @ q1_w)
            td_error_2 = target - float(feat @ q2_w)
            q1_w = q1_w + 0.03 * td_error_1 * feat
            q2_w = q2_w + 0.03 * td_error_2 * feat
            actor_grad = q1_w[2] + q1_w[3] * state + 2.0 * q1_w[4] * np.tanh(actor_w * state)
            actor_w = actor_w + 0.01 * actor_grad * state
            total = total + reward
            state = next_state
        curve.append(total)
    values = np.linspace(-1.0, 1.0, env.n_states)
    logits = np.tile(np.array([-actor_w, actor_w, -0.5 * actor_w, 0.5 * actor_w]), (env.n_states, 1))
    return logits, values, np.array(curve)


## The concept, built once: deterministic actor-critic
DDPG and TD3 use continuous actions and critic targets. The same lesson bootstrap number applies: with reward $1$, next value $0.8$, and $\gamma=0.9$, $y=1+0.9\cdot0.8=1.720$. TD3 uses the smaller twin-critic next value to reduce over-estimation.

In [ ]:

target, critic_error = deterministic_actor_critic(1.0, 0.8, 0.9)
q1_next = 1.1
q2_next = 0.8
td3_target = 1.0 + 0.9 * min(q1_next, q2_next)
state = 0.5
actor_w = 0.7
action = float(np.tanh(actor_w * state))

print("DDPG target", target)
print("critic error", critic_error)
print("TD3 clipped double target", td3_target)
print("deterministic action", action)

assert round(target, 3) == 1.720
assert round(td3_target, 3) == 1.720
assert -1.0 <= action <= 1.0


The actor is deterministic, so the policy gradient flows through $Q(s,\mu(s))$ instead of summing a discrete table. Target smoothing adds small clipped noise to the next action before evaluating twin target critics.

In [ ]:

q1_w = np.array([0.0, 0.2, 0.1, 0.0, -0.05])
q2_w = np.array([-0.02, 0.15, 0.08, 0.0, -0.08])
action, target, td_error, updated_q1 = td3_toy_update(0.5, 1.0, 0.8, actor_w, q1_w, q2_w)

print("smoothed action", action)
print("toy TD3 target", target)
print("toy TD error", td_error)
assert updated_q1.shape == q1_w.shape


## The dataset ladder: F12 continuous-control environments
D1-D4 reuse the grid ladder with a continuous-action wrapper interpretation; D5 is a tiny hand-rolled CartPole-like scalar control simulation inside `train_td3_ladder`. No gym download is used.

In [ ]:

envs = make_rl_ladder(continuous=True)
summarize_ladder(envs)

for env in envs:
    sample_state = env.reset()
    sample_next, sample_reward, sample_done = env.step(sample_state, 3, np.random.default_rng(SEED))
    print(env.name, "sample", sample_state, "->", sample_next, "reward", round(sample_reward, 3), "done", sample_done)


## Run TD3-style deterministic actor-critic across D1-D5
Each rung uses scalar continuous control features, twin critics, target smoothing, and a deterministic actor. The metric is return.

In [ ]:

results = []
artifacts = []
for env in envs:
    logits, values, curve = train_td3_ladder(env, twin=True)
    single_logits, single_values, single_curve = train_td3_ladder(env, twin=False)
    results.append((env.name, float(np.mean(curve[-10:])), float(np.mean(single_curve[-10:]))))
    artifacts.append((env, logits, values, curve, single_curve))

print("rung | td3_return | single_critic_return")
for name, td3_return, single_return in results:
    print(name, round(td3_return, 3), round(single_return, 3))


## Results visualization
The panels show value/action-field artifacts, while the summary row compares twin-critic TD3 with a single critic.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, (env, logits, values, curve, single_curve) in zip(axes[0], artifacts):
    plot_policy_panel(ax, env, values, logits, env.name)
for ax, (env, logits, values, curve, single_curve) in zip(axes[1], artifacts):
    ax.plot(curve, label="TD3 twin", color="tab:cyan")
    ax.plot(single_curve, label="single critic", color="tab:pink", alpha=0.7)
    ax.set_title("return " + env.name.split()[0])
    ax.set_xlabel("episode")
    ax.set_ylabel("return")
    ax.legend(fontsize=7)
plt.tight_layout()


## Pitfall on D5: single-critic over-estimation
Bootstrapping through one optimistic critic can inflate the target. TD3 fixes this with the minimum of twin target critics plus target smoothing.

In [ ]:

reward = 1.0
optimistic_next = 1.4
conservative_next = 0.8
single_target = reward + 0.9 * optimistic_next
twin_target = reward + 0.9 * min(optimistic_next, conservative_next)

print("single critic target", round(single_target, 3))
print("twin critic target", round(twin_target, 3))
assert single_target > twin_target
assert round(twin_target, 3) == 1.720


## Evaluate it + practice
- Metric: compare return or advantage/value error against a no-skill random-policy baseline on every rung.
- Sanity check: D1 should solve first because it has the shortest horizon and no stochastic transition.
- Ablation: turn off the key stabilizer for this lesson and the D5 metric should drop or become noisier.
- Failure signals: exploding logits, a value table with impossible magnitudes, or a policy that never reaches the goal.

Practice prompts:
1. Change $\gamma$ from 0.9 to 0.7 and explain which rungs lose the most return.
2. Turn off target smoothing and compare the target values.
3. Increase exploration noise and inspect when the deterministic actor becomes unstable.


In [ ]:
# Your code here


In [ ]:
# Your code here
